# HotpotQA-VN: dữ liệu final → graph tiếng Việt → HippoRAG2

Dùng `queries.jsonl`, `corpus.jsonl`, `qrels.tsv` từ repo Prepare-data-HotpotQA-VN. Corpus có 9.822 tài liệu; chọn ít câu hỏi KHÔNG giảm corpus. Qwen3.5-2B local, embedding multilingual E5 trên CPU, đánh giá theo qrels và answer_vi. Không dùng graph tiếng Anh cũ.

Mặc định `RUN_PHASE='prepare'` chỉ chuẩn bị dữ liệu và không cần GPU. Chọn GPU trước khi dùng `extract`, `build`, `benchmark` hoặc `all`; đổi phase lần lượt để checkpoint rõ ràng. Đọc `HOTPOTQA_VN.md` trước khi bắt đầu extraction toàn corpus.

In [ ]:
import os, sys, json, shutil, subprocess, time
from pathlib import Path
import requests
from google.colab import drive, files
drive.mount('/content/drive')
RUN_ROOT = Path('/content/drive/MyDrive/AutoSchemaKG/hotpotqa_vn_1k')
RUN_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR = RUN_ROOT / 'experiment'
OUTPUT_DIR = WORK_DIR / 'benchmark'
RUN_PHASE = 'prepare'  # prepare -> extract -> build -> package -> benchmark
MAX_QUESTIONS = 1000  # Changing this does NOT reduce the 9,822-document corpus
MODEL_ID = 'Qwen/Qwen3.5-2B'
EMBEDDING_MODEL = 'intfloat/multilingual-e5-small'
PORT = 8000
CONTEXT_LENGTH = 4096
NEEDS_LLM = RUN_PHASE in {'extract', 'build', 'benchmark', 'all'}
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if NEEDS_LLM and gpu.returncode != 0:
    raise RuntimeError('This phase needs Qwen. Select Runtime > Change runtime type > GPU.')
print(gpu.stdout if gpu.returncode == 0 else 'CPU runtime: sufficient for prepare/package')
print('Persistent experiment directory:', RUN_ROOT)

## Clone code và ghim model/code revision

Các file VN phải được push lên GitHub trước khi dùng notebook này.

In [ ]:
CODE_REF = 'codex/research-concept-retrieval'
REPO_DIR = Path('/content/SmallScaledAutoSchemaKG_vn')
if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', CODE_REF, '--single-branch',
                    'https://github.com/phuongth05/SmallScaledAutoSchemaKG.git', str(REPO_DIR)], check=True)
revision_file = RUN_ROOT / 'revisions.json'
if revision_file.exists():
    revisions = json.loads(revision_file.read_text())
    if revisions['model'] != MODEL_ID or revisions['embedding'] != EMBEDDING_MODEL:
        raise ValueError('Model changed: use a NEW RUN_ROOT')
    status = subprocess.check_output(['git', 'status', '--porcelain', '--untracked-files=no'], cwd=REPO_DIR, text=True).strip()
    if status:
        raise RuntimeError('Clone has local edits; preserve them before switching revision')
    subprocess.run(['git', 'checkout', '--detach', revisions['git_commit']], cwd=REPO_DIR, check=True)
else:
    # Also repair an older clone that was created from the repository default branch.
    subprocess.run(['git', 'fetch', 'origin', CODE_REF], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=REPO_DIR, check=True)
    def model_revision(model):
        response = requests.get(f'https://huggingface.co/api/models/{model}', timeout=30)
        response.raise_for_status()
        return response.json()['sha']
    revisions = {'git_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip(),
                 'model': MODEL_ID, 'model_revision': model_revision(MODEL_ID),
                 'embedding': EMBEDDING_MODEL, 'embedding_revision': model_revision(EMBEDDING_MODEL)}
    if not (REPO_DIR / 'scripts/run_hotpotqa_vn.py').exists():
        raise RuntimeError('v2 files are not present on this GitHub revision yet')
    revision_file.write_text(json.dumps(revisions, indent=2))
os.chdir(REPO_DIR)
print(json.dumps(revisions, indent=2))

## Clone đúng repo dữ liệu, đọc duy nhất folder final

In [ ]:
DATA_REPO = Path('/content/Prepare-data-HotpotQA-VN')
if not (DATA_REPO / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/chichic21039/Prepare-data-HotpotQA-VN.git', str(DATA_REPO)], check=True)
dataset_revision_file = RUN_ROOT / 'dataset_revision.txt'
if dataset_revision_file.exists():
    commit = dataset_revision_file.read_text().strip()
    status = subprocess.check_output(['git', 'status', '--porcelain', '--untracked-files=no'], cwd=DATA_REPO, text=True).strip()
    if status:
        raise RuntimeError('Dataset clone has local edits; preserve them before switching revision')
    subprocess.run(['git', 'checkout', '--detach', commit], cwd=DATA_REPO, check=True)
else:
    commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=DATA_REPO, text=True).strip()
    dataset_revision_file.write_text(commit)
FINAL_DIR = DATA_REPO / 'data/hotpotqa_vi_1k/final'
print('Dataset revision:', commit)
print('Reading:', FINAL_DIR)


## Môi trường tách biệt

QA/KG client dùng CPU; vLLM chỉ được cài cho phase cần LLM/GPU. Lần đầu có thể tải nhiều thư viện. Mỗi bước hiện log trực tiếp và lưu log trên Drive; lock được dùng lại khi reconnect.

In [ ]:
from collections import deque
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'uv'], check=True)
subprocess.run(['uv', '--version'], check=True)
QA_ENV = Path('/content/autoschema_qa_env')
VLLM_ENV = Path('/content/autoschema_vllm_env')
QA_PY = str(QA_ENV / 'bin/python')
VLLM_PY = str(VLLM_ENV / 'bin/python')
QA_REQUIREMENTS = REPO_DIR / 'requirements-hotpotqa-v2.txt'
COLAB_REQUIREMENTS = REPO_DIR / 'requirements-colab.txt'
required_setup_files = [QA_REQUIREMENTS, COLAB_REQUIREMENTS, REPO_DIR / 'pyproject.toml']
missing_setup_files = [path for path in required_setup_files if not path.is_file()]
if missing_setup_files:
    raise FileNotFoundError('Missing setup files: ' + ', '.join(map(str, missing_setup_files)))

def run_logged(label, command):
    log_path = RUN_ROOT / f'{label}.log'
    recent = deque(maxlen=60)
    print(f'\n=== {label} ===', flush=True)
    print('Command:', ' '.join(map(str, command)), flush=True)
    print('Log:', log_path, flush=True)
    with log_path.open('a', encoding='utf-8') as log_file:
        process = subprocess.Popen(list(map(str, command)), cwd=str(REPO_DIR),
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
            encoding='utf-8', errors='replace', bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            log_file.write(line)
            log_file.flush()
            recent.append(line.rstrip())
        return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f'{label} failed with code {return_code}. Full log: {log_path}\n'
                           + '\n'.join(recent))

if not Path(QA_PY).exists():
    subprocess.run(['uv', 'venv', '--python', '3.12', str(QA_ENV)], check=True)
qa_lock = RUN_ROOT / 'qa_requirements.lock.txt'
if qa_lock.exists() and qa_lock.stat().st_size:
    run_logged('qa_lock_install', ['uv', 'pip', 'install', '--python', QA_PY,
        '--torch-backend=cpu', '-r', qa_lock])
else:
    run_logged('qa_base_install', ['uv', 'pip', 'install', '--python', QA_PY,
        '--torch-backend=cpu', '-r', QA_REQUIREMENTS])
    run_logged('qa_repo_install', ['uv', 'pip', 'install', '--python', QA_PY,
        '--torch-backend=cpu', '--constraint', QA_REQUIREMENTS, '-r', COLAB_REQUIREMENTS])
    with qa_lock.open('w', encoding='utf-8') as lock_file:
        subprocess.run(['uv', 'pip', 'freeze', '--python', QA_PY], cwd=str(REPO_DIR),
            stdout=lock_file, text=True, check=True)

if NEEDS_LLM:
    if not Path(VLLM_PY).exists():
        subprocess.run(['uv', 'venv', '--python', '3.12', str(VLLM_ENV)], check=True)
    vllm_lock = RUN_ROOT / 'vllm_requirements.lock.txt'
    vllm_args = ['-r', vllm_lock] if vllm_lock.exists() and vllm_lock.stat().st_size else ['--pre', 'vllm']
    run_logged('vllm_install', ['uv', 'pip', 'install', '--python', VLLM_PY,
        '--torch-backend=auto', *vllm_args])
    if not vllm_lock.exists() or not vllm_lock.stat().st_size:
        with vllm_lock.open('w', encoding='utf-8') as lock_file:
            subprocess.run(['uv', 'pip', 'freeze', '--python', VLLM_PY], cwd=str(REPO_DIR),
                stdout=lock_file, text=True, check=True)
else:
    print('Skipping vLLM installation for CPU-only phase:', RUN_PHASE)

subprocess.run([QA_PY, '-c', "import torch; print('QA torch:', torch.__version__)"], check=True)


## Khởi động Qwen nếu giai đoạn cần LLM

In [ ]:
if NEEDS_LLM:
    LOG_PATH = RUN_ROOT / 'qwen_vllm.log'
    def ready():
        try:
            response = requests.get(f'http://127.0.0.1:{PORT}/v1/models', timeout=5)
            if response.ok:
                models = response.json()['data']
                if not any(m['id'] == MODEL_ID for m in models):
                    raise RuntimeError('Port is occupied by another model; change PORT')
                return True
        except requests.RequestException:
            return False
        return False
    if not ready():
        server_log = open(LOG_PATH, 'a', encoding='utf-8')
        server = subprocess.Popen([str(VLLM_ENV / 'bin/vllm'), 'serve', MODEL_ID,
            '--revision', revisions['model_revision'], '--host', '127.0.0.1', '--port', str(PORT),
            '--dtype', 'half', '--max-model-len', str(CONTEXT_LENGTH), '--max-num-seqs', '1',
            '--gpu-memory-utilization', '0.80', '--language-model-only'],
            stdout=server_log, stderr=subprocess.STDOUT)
        started = time.monotonic()
        while time.monotonic() - started < 1200:
            if server.poll() is not None:
                server_log.flush()
                print(LOG_PATH.read_text(errors='replace')[-12000:])
                raise RuntimeError('vLLM stopped; inspect log above')
            if ready():
                break
            print(f'Waiting for Qwen: {time.monotonic() - started:.0f}s', flush=True)
            time.sleep(10)
        else:
            raise TimeoutError(f'Server not ready. Inspect {LOG_PATH}; do not start another copy')
    print('Local Qwen is ready')
else:
    print('This phase does not require Qwen.')


## Chạy giai đoạn đã chọn

`prepare`: không gọi LLM. `extract`: ba bước trích xuất trên toàn corpus. `build`: sinh concept tiếng Việt và GraphML. `package`: ZIP graph/provenance. `benchmark`: 4 cấu hình, checkpoint từng câu. Nếu extraction bị ngắt giữa chừng, wrapper sẽ dừng để bạn kiểm tra; không tự xóa output hoặc giả định resume extraction.

In [ ]:
RUN_SCRIPT = REPO_DIR / 'scripts' / 'run_hotpotqa_vn.py'
if not RUN_SCRIPT.is_file():
    raise FileNotFoundError(f'Missing runner: {RUN_SCRIPT}. Check CODE_REF and rerun the clone cell.')
required_vn_files = [FINAL_DIR / name for name in ('queries.jsonl', 'corpus.jsonl', 'qrels.tsv')]
missing_vn_files = [path for path in required_vn_files if not path.is_file()]
if missing_vn_files:
    raise FileNotFoundError('Missing HotpotQA-VN final files: ' + ', '.join(map(str, missing_vn_files)))

command = [QA_PY, '-X', 'utf8', '-u', str(RUN_SCRIPT),
    '--phase', RUN_PHASE, '--source-dir', str(FINAL_DIR), '--work-dir', str(WORK_DIR),
    '--max-questions', str(MAX_QUESTIONS), '--sampling', 'random', '--seed', '42']

# prepare/package do not use either model. Add model arguments only to phases that need them.
if RUN_PHASE in {'extract', 'build', 'benchmark', 'all'}:
    command += ['--model', MODEL_ID, '--model-revision', revisions['model_revision'],
        '--base-url', f'http://127.0.0.1:{PORT}/v1', '--context-length', str(CONTEXT_LENGTH)]
if RUN_PHASE in {'benchmark', 'all'}:
    command += ['--embedding-model', EMBEDDING_MODEL,
        '--embedding-revision', revisions['embedding_revision']]

phase_log = RUN_ROOT / f'{RUN_PHASE}.log'
print('Running:', ' '.join(map(str, command)), flush=True)
print('Log:', phase_log, flush=True)
with phase_log.open('a', encoding='utf-8') as log_file:
    process = subprocess.Popen(command, cwd=str(REPO_DIR),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        encoding='utf-8', errors='replace', bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
        log_file.write(line)
        log_file.flush()
    return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f'Phase {RUN_PHASE} failed with code {return_code}. Full log: {phase_log}')
print(f'Phase {RUN_PHASE} completed.', flush=True)


## Lưu kết quả

Drive giữ input, graph, checkpoint, model/code/data revisions và dependency locks. Số liệu chỉ là kết quả hoàn chỉnh khi mọi method có `complete=true`.

In [ ]:
if (OUTPUT_DIR / 'summary.json').exists():
    print((OUTPUT_DIR / 'summary.json').read_text())
print('Saved experiment:', WORK_DIR)
# Run this cell before disconnecting; no model weights are packaged.
archive = shutil.make_archive('/content/autoschemakg_hotpotqa_vn', 'zip', RUN_ROOT)
files.download(archive)
